In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

print(os.listdir('/kaggle/input'))

In [ ]:
print(os.listdir('/kaggle/input/datasets'))

In [ ]:
print(os.listdir('/kaggle/input/datasets/kmader'))

In [ ]:
print(os.listdir('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'))

In [ ]:
import os

BASE = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'

# See all files in the dataset
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    if level < 2:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 1:
            print(f'{indent}  ({len(files)} files)')

In [ ]:
import pandas as pd

# Find metadata CSV
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith('.csv'):
            print(os.path.join(root, f))

In [ ]:
import os
import pandas as pd

# Load metadata — update path if Cell 2 shows different location
df = pd.read_csv(f'{BASE}/HAM10000_metadata.csv')
print(f"Total rows: {len(df)}")
print(df['dx'].value_counts())

# Find where images are stored
IMAGE_DIRS = []
for root, dirs, files in os.walk(BASE):
    jpgs = [f for f in files if f.endswith('.jpg')]
    if jpgs:
        IMAGE_DIRS.append(root)
        print(f"Found {len(jpgs)} images in: {root}")

In [ ]:
import shutil
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_DIR = '/kaggle/working/dataset'

LABEL_MAP = {
    'mel':   'melanoma',
    'nv':    'nevus',
    'bcc':   'basal_cell_carcinoma',
    'akiec': 'actinic_keratosis',
    'bkl':   'benign_keratosis',
    'df':    'dermatofibroma',
    'vasc':  'vascular_lesion',
}

# Create folders
for split in ['train', 'val']:
    for class_name in LABEL_MAP.values():
        os.makedirs(f'{DATASET_DIR}/{split}/{class_name}', exist_ok=True)

# Build full image path for each row
def find_image(image_id):
    for d in IMAGE_DIRS:
        path = os.path.join(d, f'{image_id}.jpg')
        if os.path.exists(path):
            return path
    return None

df['img_path'] = df['image_id'].apply(find_image)
df_found = df[df['img_path'].notna()].copy()
print(f"Images found: {len(df_found)} / {len(df)}")

# Train/val split
train_df, val_df = train_test_split(
    df_found,
    test_size=0.2,
    stratify=df_found['dx'],
    random_state=42
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Copy into folders
def copy_split(dataframe, split_name):
    for _, row in dataframe.iterrows():
        class_name = LABEL_MAP[row['dx']]
        dst = f"{DATASET_DIR}/{split_name}/{class_name}/{row['image_id']}.jpg"
        if not os.path.exists(dst):
            shutil.copy(row['img_path'], dst)
    print(f"  [{split_name}] Done")

print("Organizing images...")
copy_split(train_df, 'train')
copy_split(val_df,   'val')

# Verify
print("\nVerification:")
for split in ['train', 'val']:
    for name in LABEL_MAP.values():
        count = len(os.listdir(f'{DATASET_DIR}/{split}/{name}'))
        print(f"  {split}/{name:<30} {count}")

In [ ]:
import tensorflow as tf #TensorFlow act as the AI engine without it you cannot build or train neural networks.
from tensorflow.keras.applications import MobileNetV2 #transfer learning model
from tensorflow.keras import layers, models # Allows us to build new neural network layers.
from tensorflow.keras.preprocessing.image import ImageDataGenerator #It loads images automatically from folders
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau #ReduceLROnPlateau- Tensorflow automatically reduces the learning rate if learning slwos down
from sklearn.utils.class_weight import compute_class_weight #Used to solve class imbalance
import numpy as np

# ── Rebuild generators with LESS aggressive augmentation ──────
# Your previous augmentation was too strong for medical images
# Strong rotation/zoom distorts skin lesion features

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,        # reduced from 20
    width_shift_range=0.05,   # reduced from 0.1
    height_shift_range=0.05,  # reduced from 0.1
    shear_range=0.05,         # reduced from 0.1
    zoom_range=0.1,           # reduced from 0.2
    horizontal_flip=True,
    vertical_flip=True,       # skin lesions look same upside down
    fill_mode='nearest'
    # removed brightness_range — it was confusing color-based features
)
val_datagen = ImageDataGenerator(rescale=1./255)

DATASET_DIR = '/kaggle/working/dataset'
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 7

train_gen = train_datagen.flow_from_directory(
    f'{DATASET_DIR}/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    f'{DATASET_DIR}/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
# Calculate class weights to handle dataset imbalance
# Rare diseases get higher importance during training
class_weights_arr = compute_class_weight(
    class_weight='balanced', #automatically calculates weights 
    classes=np.unique(train_gen.classes), #removes duplicates
    y=train_gen.classes #distributes the weights according to number of images(eg- less number of images=more weight)
)

# Convert weights into dictionary format because TensorFlow expects a dictionary
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class indices:", train_gen.class_indices)
print("Class weights:", class_weight_dict)

#MobileNetV2 was initialized with weights pretrained on the ImageNet dataset, 
#which contains millions of labeled images. 
#This allows the model to reuse previously learned visual features such as edges, textures, and shapes.
#We then applied transfer learning to adapt these features for skin disease classification using the HAM10000 dataset
# Load pretrained MobileNetV2 (Transfer Learning)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),   # Image size expected by MobileNet
    include_top=False,           # Remove original ImageNet classifier
    weights='imagenet'           # Load pretrained ImageNet weights
)

# Freeze MobileNet layers
# Only our custom layers will learn initially
base_model.trainable = False

#phase 1 training - feature extraction
# Build custom classification head
#ReLU stands for:
#Rectified Linear Unit

#Its formula is:
#ReLU(x)=max(0,x)
model = models.Sequential([

    # Pretrained feature extractor
    base_model,

    # Convert feature maps into feature vector(compression)
    layers.GlobalAveragePooling2D(),

    # Normalize activations for stable training
    layers.BatchNormalization(),

    # Learn skin disease specific patterns
    layers.Dense(256, activation='relu'),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Additional feature learning layer
    layers.Dense(128, activation='relu'),

    # Further regularization
    layers.Dropout(0.3),

    # Final output layer (7 disease classes)
    layers.Dense(NUM_CLASSES, activation='softmax') #Softmax converts outputs into probabilities.
])
# Lower learning rate: 5e-4 instead of 1e-3
#val accuracy was unstable so to make it stable we do this 
# Configure training process
# Adam (Adaptive Moment Estimation)
# Uses previous gradients and momentum to update weights efficiently.
# learning_rate=0.0005 controls how much the weights change after each update.
model.compile(

    # Adam optimizer with learning rate 0.0005
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),

    # Multi-class classification loss
    loss='categorical_crossentropy',

    # Track classification accuracy
    metrics=['accuracy']
)

# Print complete model architecture
model.summary()
print("\nModel rebuilt with improved settings.")

In [ ]:
import json

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,               # increased from 5 — give it more time
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        '/kaggle/working/best_model.keras',  # .keras format (avoids warning)
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,               # more aggressive LR reduction
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Phase 1: Training with improved settings...")
print("Target: reach ~70%+ val accuracy before fine-tuning\n")

history1 = model.fit(
    train_gen,
    epochs=25,                    # more epochs, early stopping will cut short
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

# Save class indices
with open('/kaggle/working/class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)

best_val_acc = max(history1.history['val_accuracy'])
print(f"\nBest val accuracy Phase 1: {best_val_acc*100:.2f}%")

In [ ]:
# Cell A — Reload best model from this session
# (only if your session is still alive with the model in memory)
# Skip this if model variable still exists

import tensorflow as tf
import numpy as np
import json
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

# Reload saved model
model = tf.keras.models.load_model('/kaggle/working/best_model.keras')
print("Model loaded")
print("Current val accuracy baseline: ~61%")

In [ ]:
# Cell C — Aggressive fine-tuning
# This is where the big accuracy jump happens
# We unfreeze MORE layers than before (last 60 instead of 30)

# Get the MobileNetV2 base layer from inside Sequential model
base_model = model.layers[0]
base_model.trainable = True

# Unfreeze last 60 layers (more than previous 30)
# More unfrozen layers = model learns more skin-specific features
for layer in base_model.layers[:-60]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen base layers: {unfrozen} / {len(base_model.layers)}")

# Very low learning rate — critical for fine-tuning
# Too high = destroys pretrained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model_finetuned.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=4,
        min_lr=1e-8,
        verbose=1
    )
]

print("\nStarting fine-tuning...")
print("Expected: accuracy should jump to 75-83%\n")

history_ft = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks_ft
)

best_acc = max(history_ft.history['val_accuracy'])
print(f"\nBest fine-tuned val accuracy: {best_acc*100:.2f}%")